# Figure 3 — HCP-TRT functional selectivity

For each subject, uses their **L1-specific langloc fROI** as a mask, then extracts
mean z-score from each HCP-TRT task contrast. Shows language selectivity:
the language task should be highest, other cognitive domains lower.

Plots: Both hemispheres, LH, RH, and each individual Fedorenko parcel.

**Inputs** (from env vars injected by `invoke run-notebooks`):
- `OUTPUT_DATA_DIR` — pipeline output root (contains both `langlocalizer/` and `hcptrt/`)
- `SOURCE_DATA_DIR` — pipeline source root (for Fedorenko atlas labels)


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from nilearn.image import resample_to_img

OUTPUT_DIR  = Path(os.environ.get('OUTPUT_DATA_DIR', 'output_data'))
SOURCE_DIR  = Path(os.environ.get('SOURCE_DATA_DIR', 'source_data'))

LANGLOC_DIR = OUTPUT_DIR / 'langlocalizer'
HCPTRT_DIR  = OUTPUT_DIR / 'hcptrt'
FROI_DIR    = LANGLOC_DIR / 'fedorenko_frois'
ATLAS_TXT   = SOURCE_DIR / 'fedorenko' / 'allParcels-language-SN220.txt'

FIGURES_OUTPUT_DIR = Path(os.environ.get('FIGURES_OUTPUT_DIR',
                          str(OUTPUT_DIR / 'figures_manuscript')))
FIG_DIR = FIGURES_OUTPUT_DIR / 'fig3_hcptrt_selectivity'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'langloc dir : {LANGLOC_DIR}')
print(f'hcptrt dir  : {HCPTRT_DIR}')
print(f'fROI dir    : {FROI_DIR}')
print(f'Figure dir  : {FIG_DIR}')

langloc dir : /scratch/ibilgin/cneuromod.glm/output_data/langlocalizer
hcptrt dir  : /scratch/ibilgin/cneuromod.glm/output_data/hcptrt
fROI dir    : /scratch/ibilgin/cneuromod.glm/output_data/langlocalizer/fedorenko_frois
Figure dir  : /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig3_hcptrt_selectivity


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
LANGUAGE_DOMINANCE = {
    'sub-01': {'L1': 'French',  'L2': 'English'},
    'sub-02': {'L1': 'French',  'L2': 'English'},
    'sub-03': {'L1': 'English', 'L2': 'French'},
    'sub-05': {'L1': 'English', 'L2': 'French'},
}
L1_TASK = {'French': 'aliceFr', 'English': 'aliceEn'}
L1_CONTRAST = 'int-degr'

# HCP-TRT tasks: display label and the contrast to use
HCP_TASKS = [
    ('language',   'HCP\nLanguage',   'story-math'),
    ('wm',         'HCP\nWM',         'twoback-zeroback_face'),
    ('social',     'HCP\nSocial',     'mental-random'),
    ('emotion',    'HCP\nEmotion',    'Face-Shape'),
    ('relational', 'HCP\nRelational', 'relational-control'),
    ('gambling',   'HCP\nGambling',   'reward-punishment'),
    ('motor',      'HCP\nMotor',      'LH-RH'),
]

SUBJECT_COLORS = {
    'sub-01': '#E63946', 'sub-02': '#06A77D',
    'sub-03': '#457B9D', 'sub-05': '#F77F00',
}
BAR_COLORS = {
    'language': '#D4E8D8', 'wm': '#B8D4E8', 'social': '#FFE5B4',
    'emotion': '#F3E5F5', 'gambling': '#FFF9C4', 'motor': '#E1BEE7',
    'relational': '#E8D4C8',
}

In [3]:
# ---------------------------------------------------------------------------
# Atlas helpers
# ---------------------------------------------------------------------------

def load_parcel_labels():
    if not ATLAS_TXT.exists():
        print(f'[WARN] Atlas labels not found: {ATLAS_TXT}')
        return []
    with open(ATLAS_TXT) as f:
        return [l.strip() for l in f if l.strip()]


PARCEL_LABELS = load_parcel_labels()
print(f'Loaded {len(PARCEL_LABELS)} parcel labels')

Loaded 10 parcel labels


In [4]:
# ---------------------------------------------------------------------------
# Data helpers
# ---------------------------------------------------------------------------

def build_l1_froi_mask(subject, roi_type='allLANG'):
    """
    Build union fROI mask from subject-level (cross-session) top-voxel files,
    using the subject's L1 task.

    roi_type: 'allLANG' | 'LH' | 'RH' | parcel name
    Returns a Nifti1Image or None.
    """
    dom = LANGUAGE_DOMINANCE.get(subject)
    if dom is None:
        return None
    task = L1_TASK[dom['L1']]

    if roi_type == 'allLANG':
        pattern = f'{subject}_task-{task}_contrast-{L1_CONTRAST}_parcel-*_top_mean.nii.gz'
    elif roi_type in ('LH', 'RH'):
        pattern = f'{subject}_task-{task}_contrast-{L1_CONTRAST}_parcel-{roi_type}_*_top_mean.nii.gz'
    else:
        pattern = f'{subject}_task-{task}_contrast-{L1_CONTRAST}_parcel-{roi_type}_top_mean.nii.gz'

    files = sorted(FROI_DIR.glob(pattern))
    if not files:
        return None

    ref   = nib.load(str(files[0]))
    union = np.zeros(ref.shape, dtype=bool)
    for f in files:
        union |= (nib.load(str(f)).get_fdata() > 0)
    return nib.Nifti1Image(union.astype(np.uint8), ref.affine, ref.header)


def extract_hcptrt_data(roi_type='allLANG'):
    """
    For every subject, load their L1 fROI mask, then extract mean z from
    each HCP-TRT task/session.
    Returns a DataFrame.
    """
    rows = []
    subjects = sorted(LANGUAGE_DOMINANCE.keys())

    for subject in subjects:
        mask_img = build_l1_froi_mask(subject, roi_type)
        if mask_img is None:
            print(f'  [SKIP] {subject}: no L1 fROI for roi_type={roi_type}')
            continue

        voxels = int((mask_img.get_fdata() > 0).sum())
        print(f'  {subject}: {voxels} voxels in {roi_type} mask')

        for task, label, contrast in HCP_TASKS:
            subj_root = HCPTRT_DIR / subject
            if not subj_root.exists():
                continue
            for ses_dir in sorted(subj_root.glob('ses-*')):
                session = ses_dir.name
                z_path  = ses_dir / task / f'{subject}_{session}_task-{task}_contrast-{contrast}_stat-z.nii.gz'
                if not z_path.exists():
                    continue
                z_img  = nib.load(str(z_path))
                mask   = mask_img.get_fdata() > 0
                if mask.shape != z_img.shape[:3]:
                    m2   = resample_to_img(mask_img, z_img, interpolation='nearest')
                    mask = m2.get_fdata() > 0
                if mask.sum() == 0:
                    continue
                rows.append({
                    'subject':    subject,
                    'session':    session,
                    'task':       task,
                    'task_label': label,
                    'contrast':   contrast,
                    'value':      float(np.nanmean(z_img.get_fdata()[mask])),
                    'roi_type':   roi_type,
                })

    return pd.DataFrame(rows)

In [5]:
# ---------------------------------------------------------------------------
# Figure builder
# ---------------------------------------------------------------------------

def make_figure3(df, roi_name='Both Hemispheres', dpi=300):
    if df is None or len(df) == 0:
        print(f'[SKIP] No data for {roi_name}')
        return None

    np.random.seed(42)
    task_order   = [t[0] for t in HCP_TASKS]
    tasks_present = [t for t in task_order if t in df['task'].unique()]
    x_pos = np.arange(len(tasks_present)) * 0.9

    fig, ax = plt.subplots(figsize=(12, 8))

    means = [df[df['task'] == t]['value'].mean() for t in tasks_present]
    sems  = [df[df['task'] == t]['value'].sem()  for t in tasks_present]

    for i, task in enumerate(tasks_present):
        ax.bar(x_pos[i], means[i], 0.85,
               color=BAR_COLORS.get(task, '#E0E0E0'),
               alpha=0.85, edgecolor='#2C3E50', linewidth=1.8, zorder=2)

    ax.errorbar(x_pos, means, yerr=sems, fmt='none', color='#2C3E50',
                capsize=7, linewidth=2.5, capthick=2.5, zorder=3)

    for xi, task in zip(x_pos, tasks_present):
        for _, row in df[df['task'] == task].iterrows():
            jitter = np.random.uniform(-0.15, 0.15)
            ax.scatter(xi + jitter, row['value'],
                       color=SUBJECT_COLORS.get(row['subject'], '#7F8C8D'),
                       s=200, zorder=10, edgecolor='white', linewidth=2.5, alpha=0.95)

    ax.set_ylabel('Mean z-score of Language Regions', fontsize=16,
                  fontweight='600', color='#2C3E50', labelpad=12)
    ax.set_xticks(x_pos)
    labels = [next(t[1] for t in HCP_TASKS if t[0] == task) for task in tasks_present]
    ax.set_xticklabels(labels, fontsize=14, color='#34495E', fontweight='600')
    if ax.get_xticklabels():
        ax.get_xticklabels()[0].set_color('#C62828')
        ax.get_xticklabels()[0].set_fontweight('bold')

    ax.set_ylim([-4, 6.5])
    ax.set_yticks(np.arange(-4, 7, 1))
    ax.set_xlim([x_pos[0] - 0.6, x_pos[-1] + 0.6])
    ax.grid(axis='y', alpha=0.25, linestyle='--', linewidth=1, color='#BDC3C7')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axhline(0, color='#7F8C8D', linewidth=1.5, linestyle='--', alpha=0.6, zorder=1)
    ax.tick_params(labelsize=12, colors='#34495E', width=1.5, length=6)

    fig.text(0.5, 0.97, 'Task Functional Selectivity',
             ha='center', fontsize=22, fontweight='bold', color='black')
    fig.text(0.5, 0.93,
             f'{roi_name} (L1-specific fROIs) — each dot is a subject and a session',
             ha='center', fontsize=14, style='italic', color='#566573')

    subjects = sorted(df['subject'].unique())
    handles  = [plt.scatter([], [], s=200, color=SUBJECT_COLORS.get(s, '#7F8C8D'),
                             edgecolor='white', linewidth=2.5, alpha=0.95)
                for s in subjects]
    ax.legend(handles, subjects, loc='upper right', frameon=True, fontsize=13,
              fancybox=True, facecolor='white', edgecolor='#BDC3C7')

    plt.tight_layout(rect=[0, 0.02, 1, 0.90])

    safe = roi_name.replace(' ', '_').replace('/', '-')
    out_path = FIG_DIR / f'fig3_selectivity_{safe}.png'
    out_pdf  = FIG_DIR / f'fig3_selectivity_{safe}.pdf'
    fig.savefig(str(out_path), dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(str(out_pdf),  bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Saved: {out_path.name}')
    return out_path

In [6]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
hcptrt_subjects = [p.name for p in HCPTRT_DIR.glob('sub-*') if p.is_dir()]
if not hcptrt_subjects:
    print('[SKIP] No hcptrt outputs found — run `invoke run-glm --dataset hcptrt` first.')
else:
    roi_configs = [
        ('allLANG', 'Both Hemispheres'),
        ('LH',      'Left Hemisphere'),
        ('RH',      'Right Hemisphere'),
    ] + [(p, p) for p in PARCEL_LABELS]

    for roi_type, roi_name in roi_configs:
        print(f'\nExtracting: {roi_name} ...')
        df = extract_hcptrt_data(roi_type=roi_type)
        if len(df) > 0:
            csv_path = FIG_DIR / f'data_fig3_{roi_type}.csv'
            df.to_csv(csv_path, index=False)
            make_figure3(df, roi_name=roi_name)

    print(f'\nFigures saved to: {FIG_DIR}')


Extracting: Both Hemispheres ...


  sub-01: 749 voxels in allLANG mask


  sub-02: 564 voxels in allLANG mask


  sub-03: 721 voxels in allLANG mask


  sub-05: 804 voxels in allLANG mask


...

  Saved: fig3_selectivity_RH_AntTemp.png

Extracting: RH_PostTemp ...
  sub-01: 115 voxels in RH_PostTemp mask


  sub-02: 151 voxels in RH_PostTemp mask


  sub-03: 166 voxels in RH_PostTemp mask


  sub-05: 155 voxels in RH_PostTemp mask


  Saved: fig3_selectivity_RH_PostTemp.png

Figures saved to: /scratch/ibilgin/cneuromod.glm/output_data/figures_manuscript/fig3_hcptrt_selectivity
